In [1]:
import requests
import torch
import time
import psutil
import subprocess

import pandas as pd

from datasets import load_dataset

In [2]:
ds = load_dataset("cardiffnlp/tweet_eval", "irony")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 784 entries, 0 to 783
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    784 non-null    object
 1   label   784 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 12.4+ KB


In [3]:
test['label'] = test['label'].apply(lambda x: 'irony' if x == 1 else 'no irony')

labels = test['label'].unique()

test

,text,label
0,@user Can U Help?||More conservatives needed o...,no irony
1,"Just walked in to #Starbucks and asked for a ""...",irony
2,#NOT GONNA WIN,no irony
3,@user He is exactly that sort of person. Weirdo!,no irony
4,So much #sarcasm at work mate 10/10 #boring 10...,irony
...,...,...
779,"If you drag yesterday into today, your tomorro...",no irony
780,Congrats to my fav @user & her team & my birth...,no irony
781,@user Jessica sheds tears at her fan signing e...,no irony
782,#Irony: al jazeera is pro Anti - #GamerGate be...,irony


In [4]:
def get_ollama_memory_usage(port=11434):
    """
    Finds the process listening on the given port using psutil
    and returns its memory usage in bytes (RSS).
    Returns None if the process isn't found or can't be accessed.
    """
    for proc in psutil.process_iter(['pid', 'name']):
        try:
            # Call proc.connections() to see if it's listening on the desired port
            for conn in proc.connections(kind='inet'):
                if conn.laddr.port == port:
                    # Found the process that listens on port=11434
                    memory_info = proc.memory_info()
                    return memory_info.rss  # in bytes
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            pass
    
    # If no process was found
    return None

get_ollama_memory_usage()

C:\Users\Rafael\AppData\Local\Temp\ipykernel_1204\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


52846592

In [5]:
def get_gpu_memory_usage():
    """
    Returns a list of used memory (in MB) for each GPU.
    """
    # Use nvidia-smi with the --query-gpu and --format flags to get just the memory usage
    command = [
        "nvidia-smi",
        "--query-gpu=memory.used",  # You can also add memory.free, name, etc.
        "--format=csv,noheader,nounits"  # CSV output with no header or units
    ]
    try:
        output = subprocess.check_output(command)
        # Decode the output from bytes to string
        output_str = output.decode("utf-8").strip()
        # Each line corresponds to one GPU's memory usage
        usage_values = [int(x) for x in output_str.split("\n")]
        return usage_values[0]
    except subprocess.CalledProcessError as e:
        print("Error running nvidia-smi:", e)
        return []


In [6]:
def classify(text, labels):

    url = "http://localhost:11434/api/chat"

    payload = {
        "model": "llama3.2:3b",
        "messages" : [
            {"role": "system", "content": "You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling binary classification tasks based on user instructions."},
            {"role": "user", "content": f"Classify the following text based on the task: Sentiment analysis of tweets. Only respond with the label that best describe the text. The possible labels are: {', '.join(labels)}. Text: {text}"}
        ],
        "stream": False,
        "options": {
            "temperature": 0
        }
    }

    start_time = time.time()
    response = requests.post(url, json=payload)
    response_time = time.time() - start_time

    vram_usage = get_gpu_memory_usage()

    ram_usage_bytes = get_ollama_memory_usage(port=11434) / (1024 * 1024)

    response = response.json()
    total_time = response['total_duration'] / 1_000_000_000
    content = response['message']['content'].lower()

    if 'no irony' in content:
        content = 'no irony'
    elif 'irony' in content:
        content = 'irony'
    else:
        content = 'error'

    print(f"Text: {text}")
    print(f"Response: {content}")

    return content, response_time, vram_usage, ram_usage_bytes, total_time

In [8]:
# apply the classify function to the test set. create one column for each output
test[['prediction', 'response_time', 'vram_usage', 'ram_usage', 'total_time']] = test['text'].apply(lambda x: classify(x, labels)).apply(pd.Series)

C:\Users\Rafael\AppData\Local\Temp\ipykernel_1204\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


Text: Exam na jud. Merry christmas. ||#Sarcasm|#KillUsSlowly
Response: irony
Text: Woke up with my foot the size of a balloon, that's just what I needed.. #not #ouch
Response: irony
Text: itsfunny bc people think im quiet|but im just listening toeveryones convos|and figuring out ur weaknesses|todestroy u later in life #sarcasm
Response: irony
Text: Loving how dominant the SEC is #Pause ..... #Not
Response: irony
Text: Gael 'pro at presentations' Anderson #not #choke
Response: irony
Text: 'Liberated' Iraq -- 2014 One of Bloodiest Years with More than 36,000 Victims: Thank you, Western liberators #sarcasm
Response: irony
Text: @user yeah I do. But you know there's this thing called an all nighter and apparently I wanna pull one #not
Response: irony
Text: #Irony: al jazeera is pro Anti - #GamerGate because feminism, or something:
Response: irony
Text: Cook and England rudderless at the death. Seven games and we've got progressively worse! Can't wait 'till they arrive here for WC #sarcasm


In [9]:
test

,text,label,prediction,response_time,vram_usage,ram_usage,total_time
0,Exam na jud. Merry christmas. ||#Sarcasm|#Kill...,irony,irony,3.515914,4135,96.902344,1.467949
1,"Woke up with my foot the size of a balloon, th...",irony,irony,2.098387,4149,97.550781,0.054683
2,itsfunny bc people think im quiet|but im just ...,irony,irony,2.095595,4150,97.625000,0.056152
3,Loving how dominant the SEC is #Pause ..... #Not,irony,irony,2.362916,4111,98.316406,0.320797
4,Gael 'pro at presentations' Anderson #not #choke,irony,irony,2.212335,4111,97.496094,0.171573
...,...,...,...,...,...,...,...
93,@user I wonder what % of that 43% agree on wha...,no irony,irony,2.190676,4116,98.226562,0.150567
94,Once again @user is re writing history re: #Cu...,no irony,irony,2.405881,4118,98.226562,0.355718
95,@user this game is pathetic. How are they losi...,no irony,irony,2.190175,4108,98.687500,0.155727
96,Photo: Orchid Tassel Chain Mermaid Dress $39.9...,no irony,no irony,2.310440,4107,97.953125,0.267032


In [10]:
y_pred = test['prediction']
y_true = test['label']

#import acc, f1_score, precision and recall from sklearn
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)

Accuracy: 0.530612
F1 score: 0.397970
Precision: 0.757895
Recall: 0.530612


In [11]:
# get average response time, vram usage and ram usage
response_time_avg = test['response_time'].mean()
vram_usage_avg = test['vram_usage'].mean()
ram_usage_avg = test['ram_usage'].mean()
total_time_avg = test['total_time'].mean()

print(f'Average response time: {response_time_avg}')
print(f'Average VRAM usage: {vram_usage_avg}')
print(f'Average RAM usage: {ram_usage_avg}')
print(f'Average total time: {total_time_avg}')

Average response time: 2.29027927408413
Average VRAM usage: 4101.7959183673465
Average RAM usage: 98.11172672193878
Average total time: 0.24558315918367346


In [12]:
# save results to txt
with open('results/gemma_ZS_binary3.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {response_time_avg}\n')
    f.write(f'Average VRAM usage: {vram_usage_avg}\n')
    f.write(f'Average RAM usage: {ram_usage_avg}\n')
    f.write(f'Average total time: {total_time_avg}\n')